In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

# Pair RDD Operations

### Create pair RDD

In [26]:
pairRdd = spark.sparkContext.parallelize([(1, 2), (3, 4), (3, 6), (3, 6), (1, 3)])


### Transformations over a pair RDD

- **mapValues**: To apply some map transformation only on the values


In [27]:
# Values are multiplied by 10
pairRdd.mapValues(lambda x: x * 10).collect()

[(1, 20), (3, 40), (3, 60), (3, 60), (1, 30)]

In [28]:
# To return range of values
pairRdd.mapValues(lambda x: range(x)).collect()

[(1, range(0, 2)),
 (3, range(0, 4)),
 (3, range(0, 6)),
 (3, range(0, 6)),
 (1, range(0, 3))]

- **flatMapValues**: To apply some map transformation only on the values and also explode/flatten the output

In [29]:
pairRdd.flatMapValues(lambda x: range(x)).collect()

[(1, 0),
 (1, 1),
 (3, 0),
 (3, 1),
 (3, 2),
 (3, 3),
 (3, 0),
 (3, 1),
 (3, 2),
 (3, 3),
 (3, 4),
 (3, 5),
 (3, 0),
 (3, 1),
 (3, 2),
 (3, 3),
 (3, 4),
 (3, 5),
 (1, 0),
 (1, 1),
 (1, 2)]

- To return the keys

In [30]:
pairRdd.keys().collect()

[1, 3, 3, 3, 1]

- To return the values

In [31]:
pairRdd.values().collect()

[2, 4, 6, 6, 3]

- sortByKey

In [32]:
# Ascending by default
pairRdd.sortByKey().collect()

[(1, 2), (1, 3), (3, 4), (3, 6), (3, 6)]

In [33]:
# To sort in descending order
pairRdd.sortByKey(ascending=False).collect()

[(3, 4), (3, 6), (3, 6), (1, 2), (1, 3)]

- reduceByKey

In [34]:
pairRdd.reduceByKey(lambda x, y: x + y).collect()

[(1, 5), (3, 16)]

**groupByKey**:
- To group the values without performing any aggregation
- This is a costly operation as it requires more memory if the data size is huge and Potential for Out-of-Memory Errors
- Hence avoid it if there is an alternative like `reduceByKey` or `aggregateByKey`

In [35]:
pairRdd.groupByKey().collect()

[(1, <pyspark.resultiterable.ResultIterable at 0x7e438b4e58b0>),
 (3, <pyspark.resultiterable.ResultIterable at 0x7e438a6dde80>)]

In [36]:
# To convert the grouped values into a list
pairRdd.groupByKey().mapValues(list).collect()

[(1, [2, 3]), (3, [4, 6, 6])]

- **substractByKey**: Returns only those pairs from 1st RDD whose keys are not present in 2nd RDD

In [37]:
first_rdd = spark.sparkContext.parallelize([(1, 2), (3, 4), (3, 6), (3, 6), (1, 3)])
second_rdd = spark.sparkContext.parallelize([(3, 9), (2, 3)])

first_rdd.subtractByKey(second_rdd).collect()

[(1, 2), (1, 3)]

- **coGroup**: Group data from both RDDs sharing same keys

In [38]:
first_rdd.cogroup(second_rdd).collect()

[(1,
  (<pyspark.resultiterable.ResultIterable at 0x7e438a6444d0>,
   <pyspark.resultiterable.ResultIterable at 0x7e4389d02960>)),
 (2,
  (<pyspark.resultiterable.ResultIterable at 0x7e4389d02cf0>,
   <pyspark.resultiterable.ResultIterable at 0x7e4389d02a80>)),
 (3,
  (<pyspark.resultiterable.ResultIterable at 0x7e4389d028a0>,
   <pyspark.resultiterable.ResultIterable at 0x7e4389d02ea0>))]

In [39]:
# To make it readable
first_rdd.cogroup(second_rdd).mapValues(lambda x: (list(x[0]), list(x[1]))).collect()
# [(key, ([left RDD values], [right RDD values]))]

[(1, ([2, 3], [])), (2, ([], [3])), (3, ([4, 6, 6], [9]))]

# Actions over pair RDD

- **collectAsMap**: Returns dict of key value pairs, if a key is repeated it returns the latest value

In [40]:
pairRdd.collectAsMap()

{1: 3, 3: 6}

- **lookup**: Looks for the key passed and returns list of associated values

In [43]:
pairRdd.lookup(3)

[4, 6, 6]

In [44]:
# Returns blank list if key not found
pairRdd.lookup(123)

[]

- **countByKey**: Returns count of each key in dictionary format

In [ ]:
pairRdd.countByKey().items()

dict_items([(1, 2), (3, 3)])

- **keyBy**: Creates a pair RDD by applying key to the values using the logic/function passed

In [49]:
rdd1 = spark.sparkContext.parallelize([100, 2, 3, 3, 410, 3, 3, 3, 4, 104, 2])
rdd1.keyBy(lambda x: 'High' if x >= 100  else 'Low').collect()

[('High', 100),
 ('Low', 2),
 ('Low', 3),
 ('Low', 3),
 ('High', 410),
 ('Low', 3),
 ('Low', 3),
 ('Low', 3),
 ('Low', 4),
 ('High', 104),
 ('Low', 2)]

In [50]:
rdd2 = spark.sparkContext.parallelize([(100, 2), (3, 3), (410, 3), (3, 3), (4, 104)])
rdd2.keyBy(lambda x: 'High' if x[0] >= 100  else 'Low').collect()

[('High', (100, 2)),
 ('Low', (3, 3)),
 ('High', (410, 3)),
 ('Low', (3, 3)),
 ('Low', (4, 104))]

### Example use case of RDD processing

In [ ]:
linesCollection = 'apple cherry orange\norange cherry mango'.split("\n")
lines = spark.sparkContext.parallelize(linesCollection, 2)
lines.flatMap(lambda x: x.split(' ')).map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y).collect()

[('apple', 1), ('cherry', 2), ('orange', 2), ('mango', 1)]